# pankhllm trainer: from your skills and logs to a ready-to-run router

This notebook is the training system for pankhllm. Run it top to bottom in Google Colab or
local Jupyter. It ends with a folder you can deploy: the pankhllm server, your trained
decision models, your skills and a config. Every language and framework then connects by
changing one base URL.

**What gets trained.** Three small decision models of pankhllm's own, none of them an LLM:

| Model | Decides | Learns from |
|---|---|---|
| `plan` | which operation in your catalog answers a question, or none | the teacher's decisions, labelled logs, templates |
| `tool` | which tool an agent turn calls | tool calls in your logs |
| `skill` | which skill file a question needs | skill `examples:`, log rows that name a skill, the teacher |

Each is a linear model over the question's wording. It trains in seconds on a CPU, is a few
MB on disk, and decides in about 0.2 ms. A **teacher** LLM labels the questions once, offline:

| Teacher | Needs |
|---|---|
| `anthropic` | `ANTHROPIC_API_KEY` |
| `openai` | `OPENAI_API_KEY` |
| `azure` | `AZURE_OPENAI_API_KEY` and your endpoint (Azure OpenAI, or OpenAI models in Azure AI Foundry) |
| `ollama` | a GPU runtime; nothing leaves the machine |

**The short way.** Give it your skills folder, prompt files and logs; nothing else is required.
From a terminal: `python notebooks/run_trainer.py --skills my-skills --prompts agent.yaml --logs app.jsonl --teacher ollama`.
In Claude Code, ask Claude to "train pankhllm on these files" and the `pankhllm-train` skill runs it.
Add an operations catalog (`--catalog`) when you also want operation decisions.

**Steps.** 1 settings, 2 get pankhllm, 3 teacher, 4 load your skills and prompts, 5 load your logs,
6 generate more questions, 7 label, 8 mine and train, 9 evaluate, 10 export the bundle,
11 try it, 12 keep it learning.

**Privacy.** Training rows keep only the question's shape: codes, dates, numbers and quoted
text are replaced by placeholders. With a cloud teacher, the questions you label are sent to
that provider. In Colab, put keys in **Secrets** (the key icon); never paste them in a cell.

## 1. Settings

In [ ]:
import os

TEACHER = os.environ.get("PANKH_TEACHER", "ollama")        # "anthropic" | "openai" | "azure" | "ollama"
TEACHER_MODEL = os.environ.get("PANKH_TEACHER_MODEL", "")  # blank = default for that teacher
AZURE_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT", "https://YOUR-RESOURCE.openai.azure.com")

CATALOG = os.environ.get("PANKH_CATALOG", "evals/pharma-kpi/router.yaml")  # operations + typed params; "" = none
SKILLS_DIR = os.environ.get("PANKH_SKILLS", "prompts")     # skill .md files
PROMPTS = [p for p in os.environ.get("PANKH_PROMPTS", "").split(",") if p]  # agent / system prompt files
LOGS = [p for p in os.environ.get("PANKH_LOGS", "").split(",") if p]  # log files; formats in step 5
WORK = os.environ.get("PANKH_WORK", "pankh-work")          # everything this notebook writes

N_TEMPLATE_TRAIN = int(os.environ.get("PANKH_N_TRAIN", "20000"))  # pharma template generator only
N_TEMPLATE_TEST = int(os.environ.get("PANKH_N_TEST", "2000"))
AUGMENT_PER_OP = int(os.environ.get("PANKH_PER_OP", "30"))        # teacher-written questions per operation
LABEL_CONCURRENCY = int(os.environ.get("PANKH_CONCURRENCY", "4"))

DEFAULT_MODEL = {"anthropic": "claude-sonnet-5", "openai": "gpt-4o-mini", "azure": "gpt-4o-mini", "ollama": "gemma4:12b"}
TEACHER_MODEL = TEACHER_MODEL or DEFAULT_MODEL[TEACHER]
print(f"teacher: {TEACHER} / {TEACHER_MODEL}")

In [ ]:
import json, shlex, subprocess, sys, shutil, pathlib, random, csv, glob, time

IN_COLAB = "google.colab" in sys.modules

def secret(name):
    """Colab Secrets first, then the environment. Values are never printed."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            v = userdata.get(name)
            if v:
                return v
        except Exception:
            pass
    return os.environ.get(name)

def sh(cmd, check=True, quiet=False):
    """Run a shell command, stream its output, return it as text."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = []
    for line in p.stdout:
        out.append(line)
        if not quiet and " INFO " not in line and " WARN " not in line:
            print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return "".join(out)

def lines(p):
    return [json.loads(l) for l in open(p) if l.strip()]

def write(p, rows):
    with open(p, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

def split(rows, test_share=0.2, seed=7):
    rows = list(rows)
    random.Random(seed).shuffle(rows)
    cut = int(len(rows) * (1 - test_share))
    return rows[:cut], rows[cut:]

## 2. Get pankhllm

Uses the repository the notebook runs in. Otherwise it clones it and builds it with Rust; the first build in Colab takes
a few minutes. Set `PANKH_BIN` to use a binary you already have.

In [ ]:
REPO_URL = "https://github.com/singhpratech/pankhllm.git"

def find_root():
    for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        c = d / "Cargo.toml"
        if c.exists() and 'name = "pankhllm"' in c.read_text():
            return d
    return None

BIN = os.environ.get("PANKH_BIN")
BIN = str(pathlib.Path(BIN).resolve()) if BIN else None  # before changing directory
LOGS = [str(pathlib.Path(p).resolve()) for p in LOGS]
PROMPTS = [str(pathlib.Path(p).resolve()) for p in PROMPTS]
if pathlib.Path(SKILLS_DIR).exists():
    SKILLS_DIR = str(pathlib.Path(SKILLS_DIR).resolve())
if CATALOG and pathlib.Path(CATALOG).exists():
    CATALOG = str(pathlib.Path(CATALOG).resolve())
ROOT = find_root()
if ROOT is None:
    if not pathlib.Path("pankhllm").exists():
        token = secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
        subprocess.run(["git", "clone", "--depth", "1", url, "pankhllm"], check=True)
        subprocess.run(["git", "-C", "pankhllm", "remote", "set-url", "origin", REPO_URL], check=True)  # drop the token
    ROOT = pathlib.Path("pankhllm").resolve()
os.chdir(ROOT)

if not BIN:
    if shutil.which("cargo") is None and not pathlib.Path.home().joinpath(".cargo/bin/cargo").exists():
        sh("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal", quiet=True)
    os.environ["PATH"] = f"{pathlib.Path.home()}/.cargo/bin:" + os.environ["PATH"]
    sh("cargo build --release --quiet")
    BIN = str(ROOT / "target/release/pankhllm")
print(sh(f"{BIN} --version", quiet=True).strip(), "at", ROOT)

## 3. The teacher

Ollama in Colab needs a GPU runtime (Runtime, Change runtime type). Locally, start Ollama and
pull the model yourself.

In [ ]:
if TEACHER == "ollama" and IN_COLAB:
    if shutil.which("ollama") is None:
        sh("curl -fsSL https://ollama.com/install.sh | sh", quiet=True)
    subprocess.Popen("ollama serve > /tmp/ollama.log 2>&1", shell=True)
    time.sleep(5)
    sh(f"ollama pull {TEACHER_MODEL}", quiet=True)

KEY_ENV = {"anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY", "azure": "AZURE_OPENAI_API_KEY"}.get(TEACHER)
if KEY_ENV:
    k = secret(KEY_ENV)
    if not k:
        raise SystemExit(f"Add {KEY_ENV} to Colab Secrets or the environment first.")
    os.environ[KEY_ENV] = k  # read by the pankhllm process; never written to a file

PROVIDER = {
    "anthropic": {"kind": "anthropic", "api_key_env": "ANTHROPIC_API_KEY", "timeout_secs": 120},
    "openai": {"kind": "openai", "base_url": "https://api.openai.com/v1", "api_key_env": "OPENAI_API_KEY", "timeout_secs": 120},
    "azure": {"kind": "azure", "base_url": AZURE_ENDPOINT, "api_key_env": "AZURE_OPENAI_API_KEY", "timeout_secs": 120},
    "ollama": {"kind": "openai", "base_url": "http://localhost:11434/v1", "timeout_secs": 120},
}[TEACHER]
print("provider:", {k: v for k, v in PROVIDER.items() if k != "api_key_env"})

## 4. Load your skills and prompts

`SKILLS_DIR` holds your skill markdown files; `PROMPTS` lists agent definitions or system
prompts (`.md`, `.yaml`) to ship with them. In Colab, set `UPLOAD_SKILLS = True` to upload a zip.
A skill's front matter can carry `triggers:` (words that select it) and `examples:` (questions
it is for). Neither is required: the teacher labels which skill each logged question needs.
Mark agent definitions and house rules `always: true` so they are sent with every request.

```markdown
---
pankhllm:
  triggers: [call activity, trx]
  examples:
    - how is my territory doing against my peers this quarter
---
# Field KPI skill
...
```

In [ ]:
UPLOAD_SKILLS = False  # Colab: set True to upload a zip of your skills folder
W = (ROOT / WORK).resolve()
W.mkdir(parents=True, exist_ok=True)
if UPLOAD_SKILLS and IN_COLAB:
    from google.colab import files
    import zipfile, io
    for name, data in files.upload().items():
        zipfile.ZipFile(io.BytesIO(data)).extractall(W / "uploaded-skills")
    SKILLS_DIR = str(W / "uploaded-skills")

try:
    import yaml
except ImportError:
    sh(f"{sys.executable} -m pip install -q pyyaml")
    import yaml

def front_matter(path):
    text = open(path, encoding="utf-8").read()
    if path.endswith((".yaml", ".yml")):
        try:
            return (yaml.safe_load(text) or {}).get("pankhllm", {}) or {}
        except Exception:
            return {}
    if not text.startswith("---"):
        return {}
    try:
        return (yaml.safe_load(text.split("---", 2)[1]) or {}).get("pankhllm", {}) or {}
    except Exception:
        return {}

# One prompts folder: your skills plus your prompt files, as the server will see it.
SKILLS_DIR = SKILLS_DIR if os.path.isabs(SKILLS_DIR) else str(ROOT / SKILLS_DIR)
PROMPT_DIR = W / "prompts"
if PROMPT_DIR.exists():
    shutil.rmtree(PROMPT_DIR)
PROMPT_DIR.mkdir()
for f in glob.glob(f"{SKILLS_DIR}/**/*", recursive=True):
    if f.endswith((".md", ".txt", ".yaml", ".yml")) and os.path.isfile(f):
        shutil.copy(f, PROMPT_DIR / pathlib.Path(f).name)
for f in PROMPTS:
    shutil.copy(f, PROMPT_DIR / pathlib.Path(f).name)

choosable = 0
for f in sorted(PROMPT_DIR.iterdir()):
    fm = front_matter(str(f))
    choosable += not fm.get("always")
    print(f"{f.stem:24} always={bool(fm.get('always')):<5} triggers={len(fm.get('triggers', []) or []):<3} examples={len(fm.get('examples', []) or [])}")
SKILL_ROUTING = choosable >= 2
print(f"\n{choosable} choosable skills" + ("; skill routing will be trained" if SKILL_ROUTING else "; skill routing needs at least two"))

## 5. Load your logs

Put any of these in `LOGS` (comma-separated in `PANKH_LOGS`, or upload in Colab):

| Format | What is read |
|---|---|
| `.jsonl` with `question` | the question; `label` and `task` (`plan`, `tool`, `skill`) if present |
| `.jsonl` of OpenAI chat requests or request/response pairs | the last user message; a tool call in the response becomes a label (an operation label when the tool is in your catalog); `pankhllm.prompt` becomes a `skill` label |
| `.csv` | a `question` column (or the first column); `label` and `task` columns if present |
| `.txt` | one question per line |

Rows with a label are used as given. The teacher labels the rest in step 7. A fifth of every
source is kept apart for testing.

In [ ]:
UPLOAD_LOGS = False  # Colab: set True to upload log files
if UPLOAD_LOGS and IN_COLAB:
    from google.colab import files
    for name, data in files.upload().items():
        (W / name).write_bytes(data)
        LOGS.append(str(W / name))

def last_user(messages):
    for m in reversed(messages or []):
        if m.get("role") == "user":
            c = m.get("content")
            if isinstance(c, list):
                c = " ".join(p.get("text", "") for p in c if isinstance(p, dict))
            return c
    return None

def rows_from(path):
    p = pathlib.Path(path)
    if p.suffix == ".txt":
        return [{"question": l.strip()} for l in open(p, encoding="utf-8") if l.strip()]
    if p.suffix == ".csv":
        out = []
        for r in csv.DictReader(open(p, encoding="utf-8")):
            q = r.get("question") or next(iter(r.values()), "")
            row = {"question": q}
            if r.get("label"):
                row["label"] = r["label"]
                row["task"] = r.get("task") or "plan"
            out.append(row)
        return out
    out = []
    for v in lines(p):
        if "question" in v:
            out.append({k: v[k] for k in ("question", "label", "task") if k in v})
            continue
        req = v.get("request", v)
        q = last_user(req.get("messages"))
        if not q:
            continue
        resp = v.get("response") or {}
        calls = ((resp.get("choices") or [{}])[0].get("message") or {}).get("tool_calls") or []
        skill = (req.get("pankhllm") or {}).get("prompt")
        if calls:
            name = calls[0]["function"]["name"]  # a catalog operation trains the plan model
            out.append({"question": q, "label": name, "task": "plan" if name in OPS else "tool"})
        if skill and skill != "auto":
            out.append({"question": q, "label": skill.split(",")[0].strip(), "task": "skill"})
        if not calls and not skill:
            out.append({"question": q})
    return out

OPS = {o["name"] for o in (yaml.safe_load(open(ROOT / CATALOG)).get("planner") or {}).get("operations", [])} if CATALOG else set()

sources = {}
log_rows = [r for f in LOGS for r in rows_from(f)]
log_rows = [r for r in log_rows if r.get("question")]
if log_rows:
    tr, te = split(log_rows)
    write(W / "logs-train.jsonl", tr)
    write(W / "logs-test-questions.jsonl", te)
    sources["your logs"] = (W / "logs-train.jsonl", W / "logs-test-questions.jsonl")
    by = {}
    for r in log_rows:
        key = f"{r.get('task', 'plan')}:{'labelled' if 'label' in r else 'unlabelled'}"
        by[key] = by.get(key, 0) + 1
    print(f"{len(log_rows)} questions from {len(LOGS)} files: {by}")
else:
    print("no logs given; using generated questions only")

## 6. Generate more questions

Two generators, both optional:

- **Templates.** The pharma example ships `evals/pharma-kpi/gen.py`, with held-out templates
  for honest testing. Write one for your catalog the same way when you need volume.
- **Teacher-written.** `pankhllm augment` asks the teacher for many phrasings per operation
  and for questions nothing should answer.

In [ ]:
m = {"provider": "teacher", "model": TEACHER_MODEL, "context_window": 32000}
if TEACHER == "ollama":
    m["effort"] = "none"  # local reasoning models: spend the tokens on the answer
cfg = yaml.safe_load(open(ROOT / CATALOG)) if CATALOG else {"decisions": {"planner": False, "tools": True}}
cfg["providers"] = {"teacher": PROVIDER}
cfg["models"] = [{"name": "teacher", "tier": "fast", "routable": False, **m},
                 {"name": "agent", "tier": "balanced", "max_output": 400, **m}]
cfg["store"] = {"path": str(W / "train.db"), "redact_questions": True}
if CATALOG:
    cfg["planner"]["model"] = "teacher"
cfg.setdefault("decisions", {})["engine"] = "native"
cfg["prompts"] = {"dir": str(PROMPT_DIR), "auto_select": True}
CONFIG = W / "train.yaml"
yaml.safe_dump(cfg, open(CONFIG, "w"), sort_keys=False)

def pk(args, **kw):
    return sh(f"{BIN} --config {shlex.quote(str(CONFIG))} {args}", **kw)

pk("check", quiet=True)

gen = ROOT / "evals/pharma-kpi/gen.py"
if CATALOG.endswith("pharma-kpi/router.yaml") and N_TEMPLATE_TRAIN > 0:
    sh(f"{sys.executable} {gen} --count")
    sh(f"{sys.executable} {gen} {N_TEMPLATE_TRAIN} > {W}/template-train.jsonl", quiet=True)
    sh(f"{sys.executable} {gen} {N_TEMPLATE_TEST} --heldout > {W}/template-test.jsonl", quiet=True)
    sources["templates"] = (W / "template-train.jsonl", W / "template-test.jsonl")

if CATALOG and AUGMENT_PER_OP > 0:
    pk(f"augment {W}/aug-questions.jsonl --per-op {AUGMENT_PER_OP}")
    tr, te = split(lines(W / "aug-questions.jsonl"))
    write(W / "aug-train.jsonl", tr)
    write(W / "aug-test-questions.jsonl", te)
    sources["teacher-written"] = (W / "aug-train.jsonl", W / "aug-test-questions.jsonl")

for name, (tr, te) in sources.items():
    print(f"{name:16} train {sum(1 for _ in open(tr)):>6}   test {sum(1 for _ in open(te)):>6}")

## 7. Label

One pass per decision. **Operations** (with a catalog): rows with a label are used as given;
the teacher plans the rest once, nothing is executed, and plans that fail catalog validation
are dropped. **Skills** (two or more choosable skills): the teacher reads each skill's name and
opening lines and picks the one the question needs, or none. Logged tool calls are used as
given. Test questions are labelled into separate files and never reach training.

In [ ]:
GIVEN_PLAN_OR_TOOL = any("label" in r and r.get("task", "plan") in ("plan", "tool") for r in log_rows)

def tasks_for(name):
    if name != "your logs":
        return ["plan"]                      # generated questions are operation questions
    plan = ["plan"] if CATALOG or GIVEN_PLAN_OR_TOOL else []
    return plan + (["skill"] if SKILL_ROUTING else [])

TESTS = {}
for name, (tr, te) in sources.items():
    for task in tasks_for(name):
        if task == "plan" and not CATALOG:
            print(f"== {name}: recording given labels (no catalog, so no operation labelling)")
        else:
            print(f"== {name}: {task} labels")
        pk(f"label {tr} --task {task} --teacher-model teacher --concurrency {LABEL_CONCURRENCY}")
        if "questions" in te.name:
            out = te.with_name(te.name.replace("-questions", f"-{task}"))
            if (CATALOG or task != "plan") and sum(1 for _ in open(te)):
                pk(f"label {te} --task {task} --teacher-model teacher --concurrency {LABEL_CONCURRENCY} --out {out}")
            else:
                out.write_text("")
        else:
            out = te
        if sum(1 for _ in open(out)):
            TESTS[(name, task)] = out
            print(f"   test set: {sum(1 for _ in open(out))} labelled questions")

## 8. Mine and train

The miner reads every label in the store, adds the catalog and skill examples, and trains
the `plan`, `tool` and `skill` models. It holds out a fifth of the question shapes and picks
the confidence threshold that meets `decisions.target_precision` (0.98 by default). Its
report also covers latency per lane, repeat questions, vocabulary it suggests adding, and
routes it quarantined.

In [ ]:
pk(f"mine --since-hours 1000000 --apply --report {W}/mine-report.md", quiet=True)
out = pk("mine --since-hours 1000000 --json", quiet=True)
report = json.loads(out[out.index("\n{") + 1:] if not out.startswith("{") else out)
for n in report.get("native", []):
    thr = n["recommended_threshold"]
    print(f"{n['task']:6} examples {n['examples']:>7}  labels {len(n['labels']):>3}  "
          f"raw held-out accuracy {format(n['holdout_accuracy'], '.1%') if n['holdout_accuracy'] is not None else 'too few examples to hold out'}  "
          f"threshold {thr if thr is not None else 'default'}")
print("\nRaw accuracy is the classifier alone. At serve time pankhllm only acts when it is confident and the")
print("wording is familiar (and, for operations, when the question fills the operation exactly), so precision is higher.")
print(f"full report: {W}/mine-report.md")

## 9. Evaluate on held-out questions

Coverage is the share decided without any LLM; precision is how many of those were right.
Everything not decided still goes to the planner, so precision is the number to watch.

In [ ]:
for (name, task), te in TESTS.items():
    print(f"\n==================== {name}: {task} ====================")
    pk(f"decide-eval {te} --task {task} --show 8", check=False)

## 10. Export a ready-to-run bundle

`bundle/` holds everything a server needs:

| File | What it is |
|---|---|
| `pankhllm` | the server, one static binary (Linux x86-64 when built here) |
| `models.json` | the trained decision models |
| `pankhllm.yaml` | catalog, models, skills, and `decisions.models_file: models.json` |
| `prompts/` | your skills |
| `run.sh` | starts the server on port 4000 |

Before production, edit `pankhllm.yaml`: point the `teacher` and `agent` models at the LLMs
you serve with, and `planner.executor.url` at your executor. Keys stay in environment variables.

In [ ]:
B = W / "bundle"
if B.exists():
    shutil.rmtree(B)
B.mkdir()
pk(f"export-models {B}/models.json")
shutil.copy(BIN, B / "pankhllm")
shutil.copytree(PROMPT_DIR, B / "prompts", dirs_exist_ok=True)
serve_cfg = yaml.safe_load(open(CONFIG))
serve_cfg["decisions"]["models_file"] = "models.json"
serve_cfg["store"] = {"path": "state/pankhllm.db", "redact_questions": True}  # keeps learning in production
serve_cfg["prompts"] = {"dir": "prompts", "auto_select": True}
yaml.safe_dump(serve_cfg, open(B / "pankhllm.yaml", "w"), sort_keys=False)
(B / "run.sh").write_text('#!/bin/sh\ncd "$(dirname "$0")" && mkdir -p state && exec ./pankhllm --config pankhllm.yaml serve --port "${PORT:-4000}"\n')
(B / "run.sh").chmod(0o755)
shutil.make_archive(str(W / "pankhllm-bundle"), "zip", B)
for f in sorted(B.iterdir()):
    size = f.stat().st_size if f.is_file() else sum(x.stat().st_size for x in f.rglob("*") if x.is_file())
    print(f"{f.name:16} {size / 1024:10.1f} KB")
print(f"\nbundle: {W}/pankhllm-bundle.zip")
if IN_COLAB:
    from google.colab import files
    files.download(str(W / "pankhllm-bundle.zip"))

## 11. Try it from any ecosystem

pankhllm speaks the OpenAI API, so every SDK and framework works unchanged: set the base URL.
This cell starts the bundle, plus the demo executor for the pharma example, and asks a few
questions with the stock OpenAI Python SDK. The `lane` column shows who answered:
`decision:<op>` is the trained model with no LLM call, `planner:<op>` is the teacher, and a
model name is the generative agent.

| Stack | Change |
|---|---|
| Python `openai`, LangChain `ChatOpenAI` | `base_url="http://HOST:4000/v1"` |
| TypeScript `openai`, Vercel AI SDK | `baseURL: "http://HOST:4000/v1"` |
| C# Semantic Kernel, Microsoft Agent Framework, `OpenAI` .NET | `Endpoint = new Uri("http://HOST:4000/v1")` |
| Go, Java, Rust, Ruby, PHP, C, C++ | the clients in `sdks/`, or any OpenAI client with the base URL |

In [ ]:
PORT = int(os.environ.get("PANKH_PORT", "4150"))
PHARMA = CATALOG.endswith("pharma-kpi/router.yaml")
procs = []
if PHARMA:
    procs.append(subprocess.Popen([sys.executable, "evals/pharma-kpi/executor.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL))
procs.append(subprocess.Popen(f"PORT={PORT} {B}/run.sh", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL))
import urllib.request
for _ in range(50):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1)
        break
    except Exception:
        time.sleep(0.2)

def skills_in(reasons):
    for r in reasons or []:
        if r.startswith("skill/agent prompt"):
            return r.split('"')[1]
    return "-"

# Which skill each question gets: /v1/route explains a request without calling any model.
held_out = [r["question"] for r in lines(W / "logs-test-questions.jsonl")][:6] if (W / "logs-test-questions.jsonl").exists() else []
for q in held_out or ["how is my territory doing against my peers?"]:
    body = json.dumps({"messages": [{"role": "user", "content": q}]}).encode()
    d = json.load(urllib.request.urlopen(urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/route", data=body, headers={"content-type": "application/json"})))
    print(f"skills {skills_in(d.get('reasons')):<32} {q[:70]}")

if PHARMA:
    try:
        from openai import OpenAI
    except ImportError:
        sh(f"{sys.executable} -m pip install -q openai")
        from openai import OpenAI
    client = OpenAI(base_url=f"http://127.0.0.1:{PORT}/v1", api_key="local")
    print()
    for q in ["what were total scripts for Zelora in T-112 in 2025-07?", "how is D-12 doing against my peers on call reach?",
              "top 5 territories by NBRx for Cardivex last quarter", "who covers R-3?", "why did calls fall in T-112 in 2025-08?"]:
        t = time.perf_counter()
        r = client.chat.completions.create(model="auto", messages=[{"role": "user", "content": q}], max_tokens=200)
        ms = (time.perf_counter() - t) * 1000
        lane = (r.model_extra or {}).get("pankhllm", {}).get("model", r.model)
        print(f"{ms:8.1f} ms  {lane:<28} {q[:55]:<56} {(r.choices[0].message.content or '')[:60]!r}")

In [ ]:
for p in procs:  # stop the demo server and executor
    p.terminate()

## 12. Keep it learning automatically

Training does not have to be a manual step. Three ways to automate it, from least to most work:

1. **Let production teach it.** The bundle's config keeps a trace store. Every planner
   decision, every clean tool call and every request that names a skill is recorded as a
   label, shape only. Schedule `pankhllm mine --apply` nightly (cron, Windows Task Scheduler,
   a Kubernetes CronJob); the server reloads the models every 5 minutes.
2. **Run this notebook on a schedule.** It takes its settings from environment variables
   (`PANKH_TEACHER`, `PANKH_LOGS`, `PANKH_SKILLS`, ...), so it runs headless:
   `papermill train_pankhllm.ipynb out.ipynb`, or as a scheduled job in GitHub Actions,
   Azure Machine Learning, Databricks or Vertex AI. Drop new logs into a folder or bucket,
   let the job pick them up, and publish `models.json`.
3. **Ship models, not databases.** Copy the new `models.json` next to the server and call
   `POST /v1/decisions/reload`, or restart. A trace store with newer models wins over the file.

Watch precision in step 9 on every run. If it drops, raise `decisions.min_known_words` or
keep the previous `models.json`.